In [1]:
# Table 6

from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime
from dateutil import relativedelta

DATA_PATH = Path.cwd().parent.parent
results_folder =  Path.cwd().parent.parent / 'results' / 'pyannote_metrics'

# Load ider
ider_lena = pd.read_csv(results_folder / 'its_eaf_an1' / 'ider_2mn_clips.csv')
ider_vtc = pd.read_csv(results_folder / 'vtc_eaf_an1' / 'ider_2mn_clips.csv')

# Order
desired_order = ['low_risk', 'angelman_syndrome', 'fragile_x_syndrome', 'down_syndrome', 'autism_sibling']
ider_lena['group_id'] = pd.Categorical(ider_lena['group_id'], categories=desired_order, ordered=True)
ider_vtc['group_id'] = pd.Categorical(ider_vtc['group_id'], categories=desired_order, ordered=True)
ider_lena = ider_lena.sort_values(['group_id', 'recording_id']).reset_index(drop=True)
ider_vtc = ider_vtc.sort_values(['group_id', 'recording_id']).reset_index(drop=True)

# Read metadata
children = pd.read_csv(DATA_PATH / 'data' / 'metadata' / 'children.csv')
recordings = pd.read_csv(DATA_PATH / 'data' 'metadata' / 'recordings.csv')
recordings_data = recordings.merge(children, on='child_id')[['group_id', 'date_iso', 'recording_filename', 'child_sex', 'child_dob', 'child_id', 'age_months']]
recordings_data['age'] = recordings_data['age_months']


ider_lena = ider_lena.merge(recordings_data, how='left', left_on='recording_id', right_on='recording_filename', suffixes=('', '_y'))
ider_vtc = ider_vtc.merge(recordings_data, how='left', left_on='recording_id', right_on='recording_filename', suffixes=('', '_y'))

def compute_ider(ider_data):
    cols = ['missed detection', 'false alarm', 'confusion', 'correct']
    for col in cols:
        ider_data[col.replace(' ', '_')] = 100*ider_data[col]/ider_data['total']
    ider_data['ider'] *= 100
    return ider_data

ider_lena = compute_ider(ider_lena)
ider_vtc = compute_ider(ider_vtc)
nan_rows = pd.isna(ider_lena['confusion'])
ider_lena = ider_lena[~nan_rows]
ider_vtc = ider_vtc[~nan_rows]

FileNotFoundError: [Errno 2] No such file or directory: '/home/engaclew/neurogen/results/pyannote_metrics/its_eaf_an1/ider_2mn_clips.csv'

In [5]:
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning

def print_full_mixed_effects_results(data, dvs):
    """
    Print mixed effects results table for multiple dependent variables.
    """
    warnings.filterwarnings('ignore', category=ConvergenceWarning)
    
    print("\nTable X. Mixed Effects Analysis Results")
    print("-" * 100)
    print(f"{'Dependent Variable':<20} {'Predictor':<20} {'β':>8} {'SE':>8} {'z':>8} {'p':>10}")
    print("-" * 100)
    
    for dv in dvs:
        # Fit model
        model = smf.mixedlm(
            f"{dv} ~ group_id + age + child_sex", 
            data=data,
            groups="child_id"
        ).fit()
        
        # Get results table
        results = model.summary().tables[1]
        
        # Process results (skip the last row which is for random effects)
        for idx in range(len(results)-1):
            name = results.index[idx]
            beta = float(results.iloc[idx, 0])  # Coefficient (beta)
            se = float(results.iloc[idx, 1])    # Standard error
            z_stat = float(results.iloc[idx, 2])
            p_val = float(results.iloc[idx, 3])
            
            # Clean up predictor names
            clean_name = (name.replace('group_id[T.', '')
                            .replace('child_sex[T.', '')
                            .replace(']', '')
                            .replace('_', ' '))
            
            # Calculate partial eta-squared
            df_resid = model.df_resid
            eta_sq = (z_stat**2) / (z_stat**2 + df_resid)
            
            # Format p-value
            if p_val < 0.001:
                p_value = "< .001***"
            else:
                p_value = f"{p_val:.3f}"
                if p_val < 0.01:
                    p_value += "**"
                elif p_val < 0.05:
                    p_value += "*"
            
            print(f"{dv:<20} {clean_name:<20} {beta:>8.2f} {se:>8.2f} {z_stat:>8.2f} {p_value:>10}")
        
        print()
    
    print("-" * 100)
    print("Note: β = beta coefficient, SE = standard error, η²p = partial eta-squared")
    print("* p < .05, ** p < .01, *** p < .001")

# Example usage:
dvs = ['confusion', 'missed_detection', 'false_alarm', 'correct']
print('LENA')
print_full_mixed_effects_results(ider_lena, dvs)
print('ACLEW')
print_full_mixed_effects_results(ider_vtc, dvs)

LENA

Table X. Mixed Effects Analysis Results
----------------------------------------------------------------------------------------------------
Dependent Variable   Predictor                   β       SE        z          p
----------------------------------------------------------------------------------------------------
confusion            Intercept                8.82     4.42     2.00     0.046*
confusion            angelman syndrome        0.33     2.66     0.12      0.902
confusion            fragile x syndrome       2.39     2.64     0.90      0.366
confusion            down syndrome           -1.46     2.59    -0.57      0.572
confusion            autism sibling           4.53     2.64     1.71      0.087
confusion            m                       -0.03     1.71    -0.02      0.985
confusion            age                      0.19     0.19     0.98      0.327

missed_detection     Intercept               42.51     7.80     5.45  < .001***
missed_detection     angelman s

In [6]:
from scipy import stats

def compare_models_segmentation(data, dvs):
    """
    Compare models with and without diagnostic group for segmentation metrics.
    """
    warnings.filterwarnings('ignore', category=ConvergenceWarning)
    
    print("\nModel Comparison: With vs. Without Diagnostic Group (Segmentation Metrics)")
    print("-" * 120)
    print(f"{'DV':<20} {'Model':<30} {'LogLik':>12} {'AIC':>12} {'R²m':>8} {'χ²':>10} {'df':>5} {'p-value':>12} {'ΔR²':>10}")
    print("-" * 120)
    
    for dv in dvs:
        # Model without diagnostic group - USE ML NOT REML
        model_base = smf.mixedlm(
            f"{dv} ~ age + child_sex", 
            data=data,
            groups="child_id"
        ).fit(reml=False)
        
        # Model with diagnostic group - USE ML NOT REML
        model_full = smf.mixedlm(
            f"{dv} ~ group_id + age + child_sex", 
            data=data,
            groups="child_id"
        ).fit(reml=False)
        
        # Calculate marginal R² for base model
        var_fixed_base = np.var(model_base.fittedvalues)
        var_random_base = float(model_base.cov_re.iloc[0, 0])
        var_residual_base = model_base.scale
        total_var_base = var_fixed_base + var_random_base + var_residual_base
        r2_marginal_base = var_fixed_base / total_var_base
        
        # Calculate marginal R² for full model
        var_fixed_full = np.var(model_full.fittedvalues)
        var_random_full = float(model_full.cov_re.iloc[0, 0])
        var_residual_full = model_full.scale
        total_var_full = var_fixed_full + var_random_full + var_residual_full
        r2_marginal_full = var_fixed_full / total_var_full
        
        # Calculate difference in R²
        delta_r2 = r2_marginal_full - r2_marginal_base
        
        # Likelihood ratio test
        lr_stat = 2 * (model_full.llf - model_base.llf)
        df_diff = len(model_full.params) - len(model_base.params)
        p_value = stats.chi2.sf(lr_stat, df_diff)
        
        # Format p-value
        if p_value < 0.001:
            p_str = "< .001***"
        else:
            p_str = f"{p_value:.3f}"
            if p_value < 0.01:
                p_str += "**"
            elif p_value < 0.05:
                p_str += "*"
        
        # Print results
        print(f"{dv:<20} {'Base (no group)':<30} {model_base.llf:>12.2f} {model_base.aic:>12.2f} {r2_marginal_base:>8.3f}")
        print(f"{dv:<20} {'Full (with group)':<30} {model_full.llf:>12.2f} {model_full.aic:>12.2f} {r2_marginal_full:>8.3f} {lr_stat:>10.2f} {df_diff:>5} {p_str:>12} {delta_r2:>10.3f}")
        print()
    
    print("-" * 120)
    print("Note: R²m = Marginal R² (variance explained by fixed effects)")
    print("      ΔR² = Additional variance explained by adding diagnostic group")
    print("      χ² = likelihood ratio statistic; Models fitted with ML for comparison")
    print("* p < .05, ** p < .01, *** p < .001")

# Run analyses
dvs = ['confusion', 'missed_detection', 'false_alarm', 'correct']  # You can add 'confusion', 'false_alarm' if needed

print('=== LENA ===')
compare_models_segmentation(ider_lena, dvs)

print('\n=== ACLEW ===')
compare_models_segmentation(ider_vtc, dvs)

=== LENA ===

Model Comparison: With vs. Without Diagnostic Group (Segmentation Metrics)
------------------------------------------------------------------------------------------------------------------------
DV                   Model                                LogLik          AIC      R²m         χ²    df      p-value        ΔR²
------------------------------------------------------------------------------------------------------------------------
confusion            Base (no group)                    -2372.84      4755.68    0.055
confusion            Full (with group)                  -2371.40      4760.81    0.072       2.87     4        0.580      0.016

missed_detection     Base (no group)                    -2687.00      5383.99    0.051
missed_detection     Full (with group)                  -2684.66      5387.32    0.072       4.67     4        0.322      0.021

false_alarm          Base (no group)                    -4512.43      9034.86    0.010
false_alarm          F